In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import datetime
import hashlib

In [9]:
services = ['yellow', 'green']
start_year = 2015
end_year = 2025
months = list(range(1,13))
run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

In [6]:
app = (
    SparkSession.builder
    .appName('01_ingesta_parquet_raw')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)


# credenciales spark
sf_option = {
    
}

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/30 22:06:28 WARN Utils: Your hostname, Andress-MacBook-Air-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.100.156 instead (on interface en0)
26/03/30 22:06:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/30 22:06:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
def get_nyc_url(service, year, month):
    return f"https://d37ci6vzurychx.cloudfront.net/trip-data/{service}_tripdata_{year}-{month:02d}.parquet"

In [8]:
def compute_trip_nk(df, service):
    columns = ['service_type', 'VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'total_amount']
    return df.withColumn('trip_nk', F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in columns]), 256))

In [10]:
def log_audit(service, year, month, status, rows_read=0, rows_written=0, notes=""):
    audit_row= [(run_id, service, year, month, status, datetime.datetime.now(), notes)]
    schema = "run_id string, service string, year int, month int, status string, rows_read long, rows_written long, event_timestamp timestamp, notes string"
    audit_df = app.createDataFrame(audit_row, schema)
    audit_df.write.format('snowflake').options(**sf_option).option('dbtable', 'load_audit').mode('append').save()